### Creating RAG Agent Built on ReAct Architecture

In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="prism-ml/ternary-bonsai-2-27b",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)

In [ ]:
from pageindex import PageIndexClient
from pathlib import Path

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")

if not PAGEINDEX_API_KEY:
    raise ValueError("PAGEINDEX_API_KEY is missing from your .env file")

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY is missing from your .env file")

pi_client = PageIndexClient(
    index={"api_key": PAGEINDEX_API_KEY},
    chat={
        "model": "openrouter/prism-ml/ternary-bonsai-2-27b",
        "backend": {
            "api_key": os.getenv("OPENROUTER_API_KEY"),
            "base_url": "https://openrouter.ai/api/v1"
        }
    }
)

data_dir = Path("../data")

In [ ]:
print(bool(PAGEINDEX_API_KEY))

In [ ]:
pdf_files = sorted(data_dir.glob("**/*.pdf"))

print(f"Found {len(pdf_files)} PDF files")

page_indexes = {}

for pdf_file in pdf_files:
    print(f"\nIndexing: {pdf_file.name}")

    result = pi_client.submit_document(
        str(pdf_file.resolve()),
        wait=True
    )

    doc_id = result["doc_id"]

    page_indexes[str(pdf_file.relative_to(data_dir))] = doc_id

    print(f"Tree index ready: {doc_id}")

with open("pageindex_document_ids.json", "w", encoding="utf-8") as f:
    json.dump(page_indexes, f, indent=4)

In [ ]:
if not page_indexes:
    raise ValueError("No PDFs were indexed.")

document_name = next(iter(page_indexes))
doc_id = page_indexes[document_name]

print(f"Loading tree for: {document_name}")

tree_result = pi_client.get_tree(
    doc_id,
    node_summary=True
)

pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")

print(json.dumps(
    pageindex_tree[0] if pageindex_tree else {},
    indent=2
))

In [ ]:
def llm_tree_search(query: str, tree: list) -> dict:
    """Select relevant nodes using the LLM."""

    def compress(nodes):
        compressed = []

        for node in nodes:
            entry = {
                "node_id": node.get("node_id"),
                "title": node.get("title", "Untitled"),
                "page": node.get(
                    "page_index",
                    node.get("start_index", "?")
                ),
                "summary": (
                    node.get("summary")
                    or node.get("text")
                    or ""
                )[:300]
            }

            if node.get("nodes"):
                entry["children"] = compress(node["nodes"])

            compressed.append(entry)

        return compressed

    compressed_tree = compress(tree)

    prompt = f"""
You are a document retrieval assistant.

Select the tree nodes most likely to contain the answer.
Use the section titles, summaries, page references, and hierarchy.

Question:
{query}

Document tree:
{json.dumps(compressed_tree, ensure_ascii=False)}

Return ONLY valid JSON in this format:
{{
    "thinking": "Brief explanation",
    "node_list": ["node_id1", "node_id2"]
}}

Select only IDs that exist in the supplied tree.
If no relevant nodes exist, return an empty node_list.
"""

    response = llm.invoke(prompt)
    content = response.content

    if isinstance(content, list):
        content = "".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    # Handle optional Markdown code fences
    content = content.strip()

    if content.startswith("```"):
        content = content.split("\n", 1)[1]
        content = content.rsplit("```", 1)[0].strip()

    result = json.loads(content)

    if not isinstance(result.get("node_list"), list):
        raise ValueError("The LLM did not return a valid node_list.")

    return result

In [ ]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively find nodes by their IDs."""
    found = []

    for node in tree:
        if node.get("node_id") in target_ids:
            found.append(node)

        if node.get("nodes"):
            found.extend(
                find_nodes_by_ids(node["nodes"], target_ids)
            )

    return found

In [ ]:
def generate_answer(query: str, nodes: list) -> str:
    """Generate a grounded answer from retrieved tree nodes."""

    if not nodes:
        return "No relevant sections were found in the document trees."

    context_parts = []

    for node in nodes:
        title = node.get("title", "Untitled")
        page = node.get(
            "page_index",
            node.get("start_index", "?")
        )

        content = (
            node.get("text")
            or node.get("summary")
            or "No section text or summary is available."
        )

        source = node.get("_source", "Unknown document")

        context_parts.append(
            f"Source: {source}\n"
            f"Section: {title}\n"
            f"Page: {page}\n"
            f"Content:\n{content}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""
You are an expert document analyst.

Answer the question using ONLY the supplied context.

Instructions:
- Cite the document filename, section title, and page when available.
- Be clear, concise, and accurate.
- Do not invent facts or citations.
- If the context does not contain the answer, say so.
- Distinguish facts from inferences.

Question:
{query}

Context:
{context}

Answer:
"""

    response = llm.invoke(prompt)

    if isinstance(response.content, str):
        return response.content

    if isinstance(response.content, list):
        return "".join(
            item.get("text", "")
            for item in response.content
            if isinstance(item, dict)
        )

    return str(response.content)

In [ ]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex Vectorless RAG pipeline.

    Step 1: LLM Tree Search   -> finds relevant node IDs
    Step 2: Node Retrieval    -> retrieves selected nodes
    Step 3: Answer Generation -> generates a grounded answer
    """

    if verbose:
        print("=" * 55)
        print(f"🔍 Query: {query}")
        print("=" * 55)

    # Step 1: Search the tree
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve relevant nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n.get('title', 'Untitled') for n in nodes]}")

    # Step 3: Generate the answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

In [ ]:
query = "What are the main findings across these documents?"

answer = vectorless_rag(
    query=query,
    document_trees=document_trees
)